# 01 - Análisis Exploratorio de IOCs

**Objetivo:** Explorar los datasets crudos de ThreatFox, MalwareBazaar y URLhaus
para entender su estructura, calidad y utilidad para construir features.

**Datasets:**
- `threatfox.csv`
- `malwarebazaar.csv`
- `urlhaus.csv`

In [1]:
# Cell 2 - Configuración
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from collections import Counter

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
plt.rcParams["figure.figsize"] = (10, 5)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data" / "raw"
REPORTS_DIR = PROJECT_ROOT / "reports"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print("Pandas:", pd.__version__)
print("DATA_DIR:", DATA_DIR.resolve())

Pandas: 3.0.6
DATA_DIR: /mnt/e/ConquerBlocks/Proyecto JTL/threat_intelligence/ai/data/raw


In [2]:
# Cell 3 - Carga
datasets = {
    "threatfox": pd.read_csv(DATA_DIR / "threatfox.csv", encoding="utf-8", low_memory=False),
    "malwarebazaar": pd.read_csv(DATA_DIR / "malwarebazaar.csv", encoding="utf-8", low_memory=False),
    "urlhaus": pd.read_csv(DATA_DIR / "urlhaus.csv", encoding="utf-8", low_memory=False),
}

def resumen(df, nombre):
    print(f"\n{'='*60}\n{nombre}\n{'='*60}")
    print("Shape:", df.shape)
    print("Columnas:", df.columns.tolist())
    print("\nNulos (top 10):")
    print(df.isnull().sum().sort_values(ascending=False).head(10))
    print("\nDuplicados:", df.duplicated().sum())

for nombre, df in datasets.items():
    resumen(df, nombre)

ParserError: Error tokenizing data. C error: Expected 1 fields in line 9, saw 14


In [ ]:
# Cell 4 - Análisis específico ThreatFox
tf = datasets["threatfox"]
print(tf["ioc_type"].value_counts(normalize=True).mul(100).round(2))
print(tf["malware_printable"].value_counts().head(10))
print(tf["confidence_level"].describe())

In [ ]:
# Cell 5 - Análisis temporal
tf["first_seen_utc"] = pd.to_datetime(tf["first_seen_utc"], errors="coerce")
tf["last_seen_utc"]  = pd.to_datetime(tf["last_seen_utc"], errors="coerce")
print("Rango first_seen:", tf["first_seen_utc"].min(), "→", tf["first_seen_utc"].max())
print("IOCs activos últimos 30d:",
      (tf["last_seen_utc"] > pd.Timestamp.now() - pd.Timedelta(days=30)).sum())

In [ ]:
# Cell 6 - Tags
tags = tf["tags"].dropna().astype(str).str.split(",").explode().str.strip()
print(tags.value_counts().head(15))

In [ ]:
# Cell 7 - Visualizaciones
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
tf["ioc_type"].value_counts().plot(kind="bar", ax=axes[0], title="Tipos de IOC")
tf["malware_printable"].value_counts().head(10).plot(kind="barh", ax=axes[1], title="Top 10 familias")
axes[1].invert_yaxis()
plt.tight_layout()
plt.savefig(REPORTS_DIR / "01_ioc_distribucion.png", dpi=120, bbox_inches="tight")
plt.show()
plt.close(fig)

## Hallazgos

- **Total de IOCs:** X registros
- **Tipos predominantes:** domain (~%), ip:port (~%), url (~%), md5/sha256 (~%)
- **Fuentes más activas:** reporter1, reporter2, ...
- **Familias más reportadas:** ...

### Campos útiles para features
- `ioc_type`, `malware_printable`, `confidence_level`, `first_seen_utc`,
  `last_seen_utc`, `reporter`, `tags`, TLD del dominio, longitud del IOC.

### Limitaciones detectadas
- **Sesgo:** sobrerrepresentación de ciertas familias (ej. Cobalt Strike).
- **Datos expirados:** registros con `last_seen_utc` muy antiguo.
- **Nulos:** columnas como `malware_printable` o `tags` con muchos vacíos.
- **Duplicados:** IOCs repetidos entre fuentes → riesgo de leakage.